In [ ]:
import os
import librosa
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA


# 노이즈 제거 함수
def spectral_subtract_denoise(y, sr, noise_duration=0.2, n_fft=2048, hop_length=512):
    # STFT
    D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    magnitude, phase = np.abs(D), np.angle(D)

    # noise profile 계산 (처음 noise_duration 초 구간)
    noise_frames = int((noise_duration * sr) / hop_length)
    noise_profile = np.mean(magnitude[:, :noise_frames], axis=1, keepdims=True)

    # 스펙트럼 감산 (음수가 되지 않도록 max 처리)
    magnitude_denoised = np.maximum(magnitude - noise_profile, 0)

    # 위상과 다시 결합
    D_denoised = magnitude_denoised * np.exp(1j * phase)

    # 역 STFT로 시간 영역 복원
    y_denoised = librosa.istft(D_denoised, hop_length=hop_length)
    
    return y_denoised


 # 특징 추출 함수
def extract_features(y_input):
    #CQCC 특징추출
    cqt = np.abs(librosa.cqt(y_input, sr=sr, n_bins=84, bins_per_octave=12))
    cqt_power = cqt ** 2
    log_cqt = np.log(cqt_power + 1e-8)
    cqcc = dct(log_cqt, type=2, axis=0, norm='ortho')[:19].T
    cqcc_T = cqcc.T
    cfcc_delta = librosa.feature.delta(cqcc_T)
    
    #MFCC 특징추출
    mfcc = librosa.feature.mfcc(y=y_input, sr=sr, n_mfcc=19, n_mels=128, fmin=40)
    mfcc_delta = librosa.feature.delta(mfcc, width=7)
    
    # 특징 결합
    mfcc_combined = np.concatenate((cqcc_T,mfcc_delta, cfcc_delta), axis=0)
    mfcc_mean = np.mean(mfcc_combined, axis=1)
    
    return mfcc_mean


# 훈련 데이터 metadata 경로 지정
train_metadata_path = '../2501ml_data/label/train_label.txt'

# 훈련 데이터 경로 지정
train_data_path = '../2501ml_data/train'

# 훈련 데이터의 음성 파일과 레이블을 저장할 리스트 초기화
train_x = []
train_y = []

with open(train_metadata_path, 'r') as f:
    # 각 줄을 읽어와서 음성 파일 경로와 레이블을 분리
    i = 0
    for line in f:
        #진행상황
        print("\r", i,"개 훈련데이터 특징 추출중...", end="")
        i += 1
        
        # 각 줄은 화자, 파일 이름, 라벨로 구성되어 있음
        spk, file_name, _, _, label = line.strip().split(' ')
        
        # wav 경로
        wav_path = os.path.join(train_data_path, file_name)

        # MFCC, MFCC delta, MFCC delta-delta 추출
        y, sr = librosa.load(wav_path, sr=16000)
        
        # 노이즈 제거
        y = spectral_subtract_denoise(y, sr)
        
        # 특징 추출
        mfcc_mean = extract_features(y)
        train_x.append(mfcc_mean)
        train_y.append(label)

    print()

print("훈련 데이터의 개수:", len(train_x))
print("훈련 데이터 X의 shape:", train_x[0].shape)


# 리스트를 numpy 배열로 변환
train_x = np.array(train_x)
train_y = np.array(train_y)

# 전체 훈련 데이터를 기준으로 스케일링 학습
scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)

# PCA 정의: 20차원으로 축소 (필요 시 조정)
pca = PCA(n_components=40, whiten=True)

# 훈련/테스트에 동일하게 적용
train_x_reduced = pca.fit_transform(train_x_scaled)  # 정규화된 데이터 사용

# SVM 모델 정의
svm_model = SVC(kernel='rbf', C=1, gamma='scale', probability=True)

# 모델 훈련
svm_model.fit(train_x_reduced, train_y)

 3500 개 훈련데이터 특징 추출중...

In [1]:
# 모델 추출 (pickle 버전)
import pickle

with open('./model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('./model/pca.pkl', 'wb') as f:
    pickle.dump(pca, f)

with open('./model/model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

NameError: name 'scaler' is not defined